<div style="display: flex; align-items: center; width: 100%;">
  <div style="display: flex; flex-direction: column; align-items: center; justify-content: center; width: 100px; margin-right: 0px;">
    <a href="https://risklab.ai" style="border: 0; line-height: 0.5;">
      <img src="../Utils/risklab_ai.gif" width="60px" style="border: 0; margin-bottom:-10px; vertical-align: middle;"/>
    </a>
  </div>
  <div style="flex-grow: 1;">
    <h1 style="margin: 0; margin-left:0; font-weight: bold; text-align: left; font-size: 38px;">
      Tuning: leakage-aware HPO with a Deflated-Sharpe gate
    </h1>
  </div>
</div>

Hyper-parameter tuning on financial data carries two hazards: **leaky CV**
(a shuffled k-fold inflates the score under overlapping labels) and **search
overfitting** (every trial is a configuration, so the in-sample HPO score must be
deflated by the trial count). Leakage-aware HPO wires the search through
**purged** cross-validation and gates the selected model with the **Deflated
Sharpe at the HPO trial count**. The honest, characterized finding: **tuning does
not create out-of-sample edge** - this is methodology / leakage-safety, not a
performance claim.

*Baseline (naive-CV grid) -> where it breaks (leakage / search overfit) -> purged HPO + DSR gate.*

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the pinned notebook environment

using Random, Statistics, Plots
gr()
theme(:default)                            # theme-adaptive figures (legible on light & dark)
default(background_color = :transparent, background_color_inside = :transparent,
        foreground_color = "#888888", gridalpha = 0.3,
        legend_background_color = :transparent)
using RiskLabAI.Validation: leakage_aware_hpo, deflated_sharpe_gate

  Activating project at `C:\risklab\risklabai\Notebooks.jl`


## 1. Leakage-aware search over a purged cross-validator
A synthetic classification problem; every sampled configuration is scored under
`PurgedKFold` (not a leaky shuffled fold).

In [2]:
rng = MersenneTwister(11)
n = 300
y = rand(rng, 0:1, n)
x = hcat(3.0 .* y .+ randn(rng, n), randn(rng, n), randn(rng, n))
starts = collect(1:n)
hpo = leakage_aware_hpo(
    x, y, Dict(:n_trees => [20, 60, 120], :max_depth => [2, 4, 6]);
    event_starts = starts, event_ends = starts, n_trials = 6, n_splits = 5,
    embargo = 0.01, random_state = 1)
println("best params = ", hpo.best_params)
println("best purged-CV score = ", round(hpo.best_score, digits = 3),
        "   trials = ", hpo.n_trials)
println("trial scores = ", round.(hpo.trial_scores, digits = 3))

best params = Dict{Symbol, Any}(:n_trees => 60, :max_depth => 6)
best purged-CV score = 0.947   trials = 6
trial scores = [0.947, 0.947, 0.943, 0.947, 0.947, 0.947]


## 2. The decisive control: the Deflated-Sharpe gate
Deflate the selected model's out-of-sample Sharpe by the HPO-inclusive trial count
and the cross-trial Sharpe dispersion. A selection passes only if the Deflated
Sharpe exceeds 0.5.

In [3]:
rng = MersenneTwister(5)
oos_returns = 0.01 .* randn(rng, 500) .+ 0.0006     # a thin selected-model OOS track
trial_sharpe_std = 0.05
gate = deflated_sharpe_gate(oos_returns, hpo.n_trials, trial_sharpe_std)
println("observed Sharpe   = ", round(gate.observed_sharpe, digits = 3))
println("benchmark (E[max] over trials) = ", round(gate.benchmark_sharpe, digits = 3))
println("Deflated Sharpe   = ", round(gate.deflated_sharpe, digits = 3))
println("passes the gate?  = ", gate.passes,
        "   (tuning does not create edge - gate it, never trust the in-sample HPO score)")

observed Sharpe   = 0.025
benchmark (E[max] over trials) = 0.065
Deflated Sharpe   = 0.185
passes the gate?  = false   (tuning does not create edge - gate it, never trust the in-sample HPO score)


## When to use / when NOT (from `appraisals/20_verdict.md`)
**Use leakage-aware HPO (search wired through purged CV, selection gated by
PBO/DSR at the HPO trial count) as the correct, efficient, leakage-safe tuning
methodology** - preferred over grid/random for search efficiency and over naive-CV
tuning for leakage-safety. **It does not improve out-of-sample performance after
deflation** (tuning does not create edge - a documented finding); admitted as
methodology / infrastructure, not a performance claim. The Optuna TPE/CMA-ES
sampler is an optional analogue; this port uses random sampling through the purged
validator.